In [ ]:

suppressPackageStartupMessages({
  library(arrow)
  library(dplyr)
  library(tidyr)
  library(stringr)
  library(purrr)
  library(survival)
  library(data.table)
  library(mice)
})


BUCKET   <- Sys.getenv("WORKSPACE_BUCKET")
path_out <- file.path(BUCKET, "sjl/outputs")
path_inv <- file.path(BUCKET, "sjl/invariant")

MIN_CASES_COX   <- 10L
FDR_THRESHOLD   <- 0.05
MI_M            <- 5L         
MI_SEED         <- 123

EXPOSURE <- "avg_sjl_hours"


COVARS_MINIMAL <- c(
  "age_at_landmark",
  "sex",
  "deprivation_index",
  "ehr_length_years"
)

COVARS_PRIMARY <- c(
  COVARS_MINIMAL,
  "median_daily_steps",
  "avg_sleep_duration_hours",
  "smoke_smoke_ever",
  "alcohol_alcohol_occasional",
  "alcohol_alcohol_regular",
  "edu_edu_hs",
  "edu_edu_college_plus",
  "income_income_mid",
  "income_income_high"
)

COVARS_SENS_BMI   <- c(COVARS_PRIMARY, "baseline_bmi")
COVARS_SENS_APNEA <- c(COVARS_PRIMARY, "apnea_flag")

COX_MODELS <- list(
  primary    = COVARS_PRIMARY,
  sens_bmi   = COVARS_SENS_BMI,
  sens_apnea = COVARS_SENS_APNEA
)


read_gcs <- function(rel_path, base = path_out) {
  local <- paste0("/tmp/", basename(rel_path))
  system(paste("gsutil cp", file.path(base, rel_path), local))
  read_parquet(local)
}

read_inv_tsv <- function(filename) {
  local <- paste0("/tmp/", filename)
  system(paste("gsutil cp", file.path(path_inv, filename), local))
  fread(local)
}

upload_parquet <- function(df, name) {
  local    <- paste0("/tmp/", name, ".parquet")
  gcs_path <- file.path(path_out, paste0(name, ".parquet"))
  write_parquet(as_arrow_table(df), local)
  system(paste("gsutil cp", local, gcs_path))
  message("Uploaded: ", gcs_path)
}


message("\n=== Loading data ===")

logistic_primary_scan <- read_gcs("phewas_logistic_primary.parquet")
logistic_minimal_scan <- read_gcs("phewas_logistic_minimal.parquet")

hit_csv <- "/tmp/logistic_hit_phecodes.csv"
system(paste("gsutil cp", file.path(path_out, "logistic_hit_phecodes.csv"), hit_csv))
logistic_hits <- fread(hit_csv) |> as_tibble()
hit_phecodes  <- logistic_hits$phecode
n_hits        <- length(hit_phecodes)

message("  phetk primary scan: ", nrow(logistic_primary_scan), " phecodes tested")
message("  phetk minimal scan: ", nrow(logistic_minimal_scan), " phecodes tested")
message("  Logistic hit list:  ", n_hits, " phecodes (FDR < ", FDR_THRESHOLD, ")")

cohort_r <- read_gcs("fig_cohort_for_r.parquet")
message("  cohort_r: ", nrow(cohort_r), " participants")

missing_primary <- setdiff(COVARS_PRIMARY, names(cohort_r))
missing_bmi     <- setdiff(COVARS_SENS_BMI, names(cohort_r))
missing_apnea   <- setdiff(COVARS_SENS_APNEA, names(cohort_r))
if (length(c(missing_primary, missing_bmi, missing_apnea)) > 0) {
  stop(
    "FATAL: Missing covariate columns in cohort_r:\n",
    "  primary missing:    ", paste(missing_primary, collapse = ", "), "\n",
    "  sens_bmi missing:   ", paste(missing_bmi,     collapse = ", "), "\n",
    "  sens_apnea missing: ", paste(missing_apnea,   collapse = ", "), "\n",
    "Re-run sjl_pipeline.py to regenerate fig_cohort_for_r.parquet."
  )
}

phecode_counts_time <- read_inv_tsv("phecode_counts_with_time.tsv")
message("  phecode_counts_time: ", nrow(phecode_counts_time), " rows")

build_cox_data <- function(pc) {
  pc_data <- phecode_counts_time |>
    filter(phecode == pc) |>
    select(person_id, phecode_time_to_event, first_event_date) |>
    mutate(
      phecode_time_to_event = as.numeric(phecode_time_to_event),
      first_event_date      = as.Date(as.character(first_event_date))
    )

  cohort_r |>
    left_join(pc_data, by = "person_id") |>
    mutate(
      incident  = as.integer(!is.na(phecode_time_to_event) & phecode_time_to_event > 0),
      time      = if_else(
        is.na(phecode_time_to_event) | phecode_time_to_event <= 0,
        censoring_time_years,
        phecode_time_to_event
      ),
      prevalent = !is.na(first_event_date) &
        !is.na(phecode_time_to_event) &
        phecode_time_to_event <= 0
    ) |>
    filter(!prevalent, time > 0, !is.na(avg_sjl_hours))
}


message("\n=== Targeted Cox validation (", n_hits,
        " phecodes × ", length(COX_MODELS), " models) ===")

fit_one_cox <- function(pc, model_name, covars) {

  df_base  <- build_cox_data(pc)
  n_events <- sum(df_base$incident, na.rm = TRUE)
  if (n_events < MIN_CASES_COX) return(NULL)

  covars_use <- covars[covars %in% names(df_base)]
  if (length(covars_use) < length(covars)) {
    message("    Dropping missing covars [", pc, "/", model_name, "]: ",
            paste(setdiff(covars, names(df_base)), collapse = ", "))
  }

  formula_str <- paste0(
    "Surv(time, incident) ~ ", EXPOSURE, " + ",
    paste(covars_use, collapse = " + ")
  )

    
  if (model_name != "sens_bmi") {
    return(
      tryCatch({
        fit <- coxph(as.formula(formula_str), data = df_base, ties = "efron")
        s   <- summary(fit)$coefficients
        ci  <- confint(fit)

        tibble(
          phecode     = pc,
          model       = model_name,
          log_hr      = s[EXPOSURE, "coef"],
          se          = s[EXPOSURE, "se(coef)"],
          HR          = exp(s[EXPOSURE, "coef"]),
          HR_lo       = exp(ci[EXPOSURE, "2.5 %"]),
          HR_hi       = exp(ci[EXPOSURE, "97.5 %"]),
          p_val       = s[EXPOSURE, "Pr(>|z|)"],
          n_events    = n_events,
          n_total     = nrow(df_base),
          concordance = summary(fit)$concordance["C"]
        )
      }, error = function(e) {
        message("    Cox failed [", pc, "/", model_name, "]: ", e$message)
        NULL
      })
    )
  }


  tryCatch({

    df_imp_base <- df_base |>
      select(
        time, incident,
        avg_sjl_hours,
        baseline_bmi,
        all_of(COVARS_PRIMARY)
      ) |>
      mutate(
        sex = as.integer(sex)
      )

    ini  <- mice(df_imp_base, maxit = 0, printFlag = FALSE)
    meth <- ini$method
    pred <- ini$predictorMatrix

    meth[] <- ""
    meth["baseline_bmi"] <- "pmm"

    imp_local <- mice(
      df_imp_base,
      method = meth,
      predictorMatrix = pred,
      m = MI_M,
      seed = MI_SEED,
      printFlag = FALSE
    )

    fit_list <- with(
      imp_local,
      coxph(
        Surv(time, incident) ~ avg_sjl_hours +
          baseline_bmi +
          age_at_landmark + sex + deprivation_index + ehr_length_years +
          median_daily_steps + avg_sleep_duration_hours +
          smoke_smoke_ever +
          alcohol_alcohol_occasional + alcohol_alcohol_regular +
          edu_edu_hs + edu_edu_college_plus +
          income_income_mid + income_income_high,
        ties = "efron"
      )
    )

    pooled <- pool(fit_list)
    summ   <- summary(pooled, conf.int = TRUE, conf.level = 0.95)

    sjl_row <- summ[summ$term == "avg_sjl_hours", , drop = FALSE]
    if (nrow(sjl_row) != 1) stop("Could not find avg_sjl_hours term in pooled results.")

    tibble(
      phecode     = pc,
      model       = model_name,
      log_hr      = sjl_row$estimate,
      se          = sjl_row$std.error,
      HR          = exp(sjl_row$estimate),
      HR_lo       = exp(sjl_row$`2.5 %`),  
      HR_hi       = exp(sjl_row$`97.5 %`), 
      p_val       = sjl_row$p.value,
      n_events    = n_events,
      n_total     = nrow(df_base),
      concordance = NA_real_
    )

  }, error = function(e) {
    message("    MI Cox failed [", pc, "/", model_name, "]: ", e$message)
    NULL
  })
}

cox_raw <- map_dfr(hit_phecodes, function(pc) {
  map_dfr(names(COX_MODELS), ~ fit_one_cox(pc, .x, COX_MODELS[[.x]]))
})

cox_all <- cox_raw |>
  left_join(
    logistic_hits |> select(phecode, phecode_string, phecode_category),
    by = "phecode"
  ) |>
  mutate(
    nominal_sig = p_val < 0.05
  ) |>
  arrange(model, p_val)

for (m in names(COX_MODELS)) {
  df_m  <- cox_all |> filter(model == m)
  fname <- paste0("cox_", m, "_results")
  upload_parquet(df_m, fname)
  message("  ", m, ": ", nrow(df_m), " phecodes | ",
          sum(df_m$nominal_sig, na.rm = TRUE),  " Nominally sig (p < 0.05)")
}

message("\n=== 2D risk surface Cox (Fig 6) ===")

top_metabolic <- cox_all |>
  filter(model == "primary", !is.na(HR),
         str_detect(str_to_lower(phecode_category),
                    "endocrin|metabol|diabet|obes")) |>
  slice_min(p_val, n = 1)

if (nrow(top_metabolic) == 0) {
  top_metabolic <- cox_all |>
    filter(model == "primary", !is.na(HR)) |>
    slice_min(p_val, n = 1)
}

top_phecode_2d       <- top_metabolic$phecode
top_phecode_label_2d <- top_metabolic$phecode_string
message("  Top phecode for 2D surface: ", top_phecode_2d,
        " (", top_phecode_label_2d, ")")

pc_top_2d <- phecode_counts_time |>
  filter(phecode == top_phecode_2d) |>
  select(person_id, phecode_time_to_event, first_event_date) |>
  mutate(phecode_time_to_event = as.numeric(phecode_time_to_event),
         first_event_date      = as.Date(as.character(first_event_date)))

outcome_2d <- cohort_r |>
  left_join(pc_top_2d, by = "person_id") |>
  mutate(
    incident  = as.integer(!is.na(phecode_time_to_event) & phecode_time_to_event > 0),
    time      = if_else(is.na(phecode_time_to_event) | phecode_time_to_event <= 0,
                        censoring_time_years, phecode_time_to_event),
    prevalent = !is.na(first_event_date) &
      !is.na(phecode_time_to_event) & phecode_time_to_event <= 0
  ) |>
  filter(!prevalent, !is.na(avg_sjl_hours), !is.na(avg_sleep_duration_hours),
         !is.na(age_at_landmark), time > 0)

cox_2d_fit <- coxph(
  Surv(time, incident) ~
    avg_sjl_hours * avg_sleep_duration_hours +
    age_at_landmark + sex + deprivation_index + ehr_length_years + apnea_flag,
  data = outcome_2d, ties = "efron"
)

pred_grid <- expand.grid(
  avg_sjl_hours            = seq(0, 3.5, length.out = 60),
  avg_sleep_duration_hours = seq(5, 9.5, length.out = 60)
) |>
  mutate(
    age_at_landmark   = median(outcome_2d$age_at_landmark,   na.rm = TRUE),
    sex               = 0L,
    deprivation_index = median(outcome_2d$deprivation_index, na.rm = TRUE),
    ehr_length_years  = median(outcome_2d$ehr_length_years,  na.rm = TRUE),
    apnea_flag        = 0L
  )

pred_grid$predicted_lp  <- predict(cox_2d_fit, newdata = pred_grid, type = "lp")
pred_grid$relative_risk <- exp(pred_grid$predicted_lp - median(pred_grid$predicted_lp))
pred_grid$phecode_label <- top_phecode_label_2d

upload_parquet(as_tibble(pred_grid), "cox_2d_pred_grid")
message("  Uploaded cox_2d_pred_grid.parquet (", nrow(pred_grid), " grid points)")

message("\n=== Stratified Cox by race (Fig 7) ===")

top5_phecodes <- cox_all |>
  filter(model == "primary", !is.na(HR)) |>
  slice_min(p_val, n = 5) |>
  pull(phecode)

top5_labels <- cox_all |>
  filter(model == "primary", phecode %in% top5_phecodes) |>
  select(phecode, phecode_string)

race_groups <- c(
  "race_race_white"    = "race_white",
  "race_race_black"    = "race_black",
  "race_race_hispanic" = "race_hispanic",
  "race_race_asian"    = "race_asian",
  "race_race_other"    = "race_other"
)

fit_strat_cox <- function(pc, race_dummy_col) {
  race_label <- race_groups[[race_dummy_col]]

  pc_data <- phecode_counts_time |>
    filter(phecode == pc) |>
    select(person_id, phecode_time_to_event, first_event_date) |>
    mutate(phecode_time_to_event = as.numeric(phecode_time_to_event),
           first_event_date      = as.Date(as.character(first_event_date)))

  df <- cohort_r |>
    left_join(pc_data, by = "person_id") |>
    mutate(
      incident  = as.integer(!is.na(phecode_time_to_event) & phecode_time_to_event > 0),
      time      = if_else(is.na(phecode_time_to_event) | phecode_time_to_event <= 0,
                          censoring_time_years, phecode_time_to_event),
      prevalent = !is.na(first_event_date) &
        !is.na(phecode_time_to_event) & phecode_time_to_event <= 0
    ) |>
    filter(!prevalent,
           .data[[race_dummy_col]] == 1,
           !is.na(avg_sjl_hours), !is.na(age_at_landmark),
           time > 0)

  if (sum(df$incident, na.rm = TRUE) < MIN_CASES_COX) return(NULL)

  covars_use <- COVARS_PRIMARY[COVARS_PRIMARY %in% names(df)]

  tryCatch({
    fit <- coxph(
      as.formula(paste0("Surv(time, incident) ~ ", EXPOSURE, " + ",
                        paste(covars_use, collapse = " + "))),
      data = df, ties = "efron"
    )
    s  <- summary(fit)$coefficients
    ci <- confint(fit, level = 0.95)
    tibble(
      phecode  = pc,
      race     = race_label,
      HR       = exp(s[EXPOSURE, "coef"]),
      HR_lo    = exp(ci[EXPOSURE, "2.5 %"]),
      HR_hi    = exp(ci[EXPOSURE, "97.5 %"]),
      p_val    = s[EXPOSURE, "Pr(>|z|)"],
      n_events = sum(df$incident)
    )
  }, error = function(e) {
    message("    Strat Cox failed [", pc, "/", race_dummy_col, "]: ", e$message)
    NULL
  })
}

stratified_results <- map_dfr(top5_phecodes, function(pc) {
  map_dfr(names(race_groups), ~ fit_strat_cox(pc, .x))
}) |>
  left_join(top5_labels, by = "phecode")

upload_parquet(stratified_results, "cox_stratified_race")
message("  Uploaded cox_stratified_race.parquet (", nrow(stratified_results), " rows)")

message("\n=== Building concordance table ===")

cox_primary_res <- cox_all |> filter(model == "primary")

concordance_tbl <- logistic_primary_scan |>
  select(phecode, phecode_string, phecode_category,
         logistic_OR  = odds_ratio,
         logistic_p   = p_val,
         logistic_fdr = fdr_q,
         logistic_n   = n_cases) |>
  inner_join(
    cox_primary_res |> select(
      phecode,
      cox_HR      = HR,
      cox_HR_lo   = HR_lo,
      cox_HR_hi   = HR_hi,
      cox_p       = p_val,
      cox_n       = n_events,
      cox_nom_sig = nominal_sig
    ),
    by = "phecode"
  ) |>
  mutate(
    sig_cat = case_when(
      logistic_fdr < FDR_THRESHOLD & cox_p < 0.05 ~ "Logistic FDR & Cox Nominal sig",
      logistic_fdr < FDR_THRESHOLD                ~ "Logistic FDR sig only",
      cox_p < 0.05                                ~ "Cox Nominal sig only",
      TRUE                                        ~ "Neither"
    )
  )

upload_parquet(concordance_tbl, "concordance_logistic_cox")
message("  Concordance table: ", nrow(concordance_tbl), " phecodes")
message("  Logistic FDR & Cox Nominal sig: ",
        sum(concordance_tbl$sig_cat == "Logistic FDR & Cox Nominal sig"), " phecodes")

message("\n=== ANALYSIS NOTEBOOK COMPLETE ===")
message("Outputs → ", path_out)
message("  cox_primary_results.parquet")
message("  cox_sens_bmi_results.parquet")
message("  cox_sens_apnea_results.parquet")
message("  cox_2d_pred_grid.parquet")
message("  cox_stratified_race.parquet")
message("  concordance_logistic_cox.parquet")
message("\nNext: run sjl_figures.R (Notebook 4)")